# Nhận diện khuôn mặt theo thời gian thực

Notebook này dùng để nhận diện khuôn mặt real-time từ webcam.

## 1. Cài đặt thư viện

In [ ]:
!pip install opencv-python
!pip install face-recognition
!pip install numpy
!pip install pillow

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/face_attendance'

import os
os.chdir(PROJECT_PATH)

## 3. Import thư viện

In [ ]:
import cv2
import pickle
import numpy as np
import face_recognition
from google.colab.patches import cv2_imshow
from IPython.display import display, Javascript
from google.colab import output
from base64 import b64decode

## 4. Load mô hình

In [ ]:
MODEL_PATH = 'models/face_recognition_model.pkl'

# Load mô hình
with open(MODEL_PATH, 'rb') as f:
    data = pickle.load(f)

model = data['model']
label_encoder = data['label_encoder']

print(f"✓ Đã load mô hình")
print(f"Số lượng người: {len(label_encoder.classes_)}")
print(f"Danh sách: {label_encoder.classes_}")

## 5. Hàm chụp ảnh từ webcam

In [ ]:
def take_photo(filename='photo.jpg', quality=0.8):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Chụp ảnh để nhận diện';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data = output.eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    
    with open(filename, 'wb') as f:
        f.write(binary)
    
    return filename

## 6. Hàm nhận diện khuôn mặt

In [ ]:
def recognize_face(image_path, threshold=0.6):
    """
    Nhận diện khuôn mặt trong ảnh
    """
    # Đọc ảnh
    image = cv2.imread(image_path)
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Phát hiện khuôn mặt
    face_locations = face_recognition.face_locations(rgb_image)
    face_encodings = face_recognition.face_encodings(rgb_image, face_locations)
    
    results = []
    
    for (top, right, bottom, left), encoding in zip(face_locations, face_encodings):
        # Dự đoán
        prediction = model.predict([encoding])[0]
        probabilities = model.predict_proba([encoding])[0]
        confidence = probabilities[prediction]
        
        if confidence >= threshold:
            name = label_encoder.inverse_transform([prediction])[0]
        else:
            name = "Unknown"
        
        results.append({
            'name': name,
            'confidence': confidence,
            'location': (top, right, bottom, left)
        })
        
        # Vẽ khung
        color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
        cv2.rectangle(image, (left, top), (right, bottom), color, 2)
        
        # Vẽ text
        text = f"{name} ({confidence:.2f})"
        cv2.rectangle(image, (left, bottom - 35), (right, bottom), color, cv2.FILLED)
        cv2.putText(image, text, (left + 6, bottom - 6),
                   cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
    
    return image, results

## 7. Chạy nhận diện

In [ ]:
print("Nhấn nút để chụp ảnh và nhận diện...")

# Chụp ảnh
photo_path = take_photo('test_recognition.jpg')

# Nhận diện
result_image, results = recognize_face(photo_path, threshold=0.6)

# Hiển thị kết quả
print(f"\nPhát hiện {len(results)} khuôn mặt:")
for i, result in enumerate(results, 1):
    print(f"{i}. {result['name']} - Độ tin cậy: {result['confidence']:.2%}")

# Hiển thị ảnh
cv2_imshow(result_image)

## 8. Nhận diện liên tục

In [ ]:
# Nhận diện nhiều lần
num_tests = int(input("Số lần muốn test (mặc định 3): ") or "3")

for i in range(num_tests):
    print(f"\n{'='*50}")
    print(f"Test lần {i+1}/{num_tests}")
    print('='*50)
    
    # Chụp ảnh
    photo_path = take_photo(f'test_{i}.jpg')
    
    # Nhận diện
    result_image, results = recognize_face(photo_path)
    
    # Hiển thị kết quả
    if results:
        print(f"\nPhát hiện {len(results)} khuôn mặt:")
        for j, result in enumerate(results, 1):
            print(f"  {j}. {result['name']} - {result['confidence']:.2%}")
        cv2_imshow(result_image)
    else:
        print("Không phát hiện khuôn mặt")
    
    # Xóa file tạm
    os.remove(photo_path)

print("\n✓ Hoàn thành test nhận diện")